# 1. PREPROCESSING DATA

**Ekstraksi Kata Kunci dengan N-Gram dan IndoBERT untuk Rekomendasi Wisata Bogor**

---

In [1]:
import pandas as pd
import numpy as np
import re
import os
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

DATA_SOURCE = '../flask_api/data/bogor_tourism_data.csv'
OUTPUT_PATH = './data/'
os.makedirs(OUTPUT_PATH, exist_ok=True)

print("✅ Libraries imported!")

✅ Libraries imported!


## 1.1 Load Dataset

In [2]:
df_raw = pd.read_csv(DATA_SOURCE)

raw_info = pd.DataFrame({
    'Keterangan': ['Total Data Awal', 'Jumlah Kolom', 'Nama Kolom'],
    'Nilai': [len(df_raw), len(df_raw.columns), str(list(df_raw.columns))]
})
print("TABEL: INFO DATASET AWAL")
display(raw_info)

TABEL: INFO DATASET AWAL


,Keterangan,Nilai
0,Total Data Awal,430
1,Jumlah Kolom,7
2,Nama Kolom,"['nama', 'kategori', 'url', 'url_gambar', 'lik..."


In [3]:
# Hapus duplikat
df = df_raw.drop_duplicates(subset=['nama'], keep='first').reset_index(drop=True)

# Distribusi kategori
cat_dist = df['kategori'].value_counts().reset_index()
cat_dist.columns = ['Kategori', 'Jumlah']
cat_dist['Persentase'] = (cat_dist['Jumlah'] / len(df) * 100).round(2).astype(str) + '%'

print("TABEL: DISTRIBUSI KATEGORI")
display(cat_dist)

TABEL: DISTRIBUSI KATEGORI


,Kategori,Jumlah,Persentase
0,Arena,82,27.7%
1,Alam,82,27.7%
2,Rekreasi,62,20.95%
3,Kuliner,38,12.84%
4,Olahraga,11,3.72%
5,Seni Budaya,11,3.72%
6,Belanja,10,3.38%


## 1.2 Text Preprocessing

In [4]:
def preprocess_text(text):
    if pd.isna(text) or text == '':
        return ''
    text = str(text).lower()                      # Case folding
    text = re.sub(r'http\S+|www\S+', '', text)    # Remove URLs
    text = re.sub(r'\d+', '', text)               # Remove numbers
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)      # Remove special chars
    text = re.sub(r'\s+', ' ', text).strip()      # Remove extra spaces
    return text

df['deskripsi_clean'] = df['deskripsi'].apply(preprocess_text)

preprocess_steps = pd.DataFrame({
    'Langkah': ['1. Case Folding', '2. Remove URLs', '3. Remove Numbers', '4. Remove Special Chars', '5. Remove Extra Spaces'],
    'Deskripsi': ['Konversi ke huruf kecil', 'Hapus URL (http/www)', 'Hapus angka', 'Hapus karakter spesial', 'Hapus spasi berlebih']
})
print("TABEL: LANGKAH PREPROCESSING")
display(preprocess_steps)

TABEL: LANGKAH PREPROCESSING


,Langkah,Deskripsi
0,1. Case Folding,Konversi ke huruf kecil
1,2. Remove URLs,Hapus URL (http/www)
2,3. Remove Numbers,Hapus angka
3,4. Remove Special Chars,Hapus karakter spesial
4,5. Remove Extra Spaces,Hapus spasi berlebih


In [5]:
# Before-After Table
before_after = []
for i in range(min(5, len(df))):
    before_after.append({
        'Nama Wisata': df.iloc[i]['nama'],
        'Deskripsi Asli (100 char)': df.iloc[i]['deskripsi'][:100] + '...' if len(str(df.iloc[i]['deskripsi'])) > 100 else df.iloc[i]['deskripsi'],
        'Deskripsi Bersih (100 char)': df.iloc[i]['deskripsi_clean'][:100] + '...' if len(df.iloc[i]['deskripsi_clean']) > 100 else df.iloc[i]['deskripsi_clean']
    })

ba_df = pd.DataFrame(before_after)
print("TABEL: HASIL PREPROCESSING (Before - After)")
display(ba_df)

TABEL: HASIL PREPROCESSING (Before - After)


,Nama Wisata,Deskripsi Asli (100 char),Deskripsi Bersih (100 char)
0,Curug Ciampea,"Curug Ciampea Bogor, adalah salah – satu peson...",curug ciampea bogor adalah salah satu pesona a...
1,Bukit Cirimpak,Bukit Cirimpak salah satu camping ground yang ...,bukit cirimpak salah satu camping ground yang ...
2,Lembah Tepus,Lembah Tepus adalah destinasi wisata berupa su...,lembah tepus adalah destinasi wisata berupa su...
3,Sun Water Park Kahuripan,Sun Water Park Kahuripan menawarkan kawasan wi...,sun water park kahuripan menawarkan kawasan wi...
4,Lembah Pinus Camp & Café,Lembah Pinus Camp & Cafe merupakan salah satu ...,lembah pinus camp cafe merupakan salah satu wi...


## 1.3 Simpan Data

In [6]:
# Simpan data preprocessed (tanpa filter)
df.to_csv(f'{OUTPUT_PATH}data_preprocessed.csv', index=False, encoding='utf-8-sig')

summary_info = pd.DataFrame({
    'Tahap': ['Data Awal (Raw)', 'Setelah Hapus Duplikat', 'Data Final', 'Jumlah Kategori'],
    'Jumlah': [len(df_raw), len(df), len(df), df['kategori'].nunique()]
})
print("TABEL: SUMMARY DATA")
display(summary_info)

TABEL: SUMMARY DATA


,Tahap,Jumlah
0,Data Awal (Raw),430
1,Setelah Hapus Duplikat,296
2,Data Final,296
3,Jumlah Kategori,7


In [7]:
# File saved
saved = pd.DataFrame({
    'File': ['data_preprocessed.csv'],
    'Path': [f'{OUTPUT_PATH}data_preprocessed.csv'],
    'Rows': [len(df)]
})
print("TABEL: FILE SAVED")
display(saved)

TABEL: FILE SAVED


,File,Path,Rows
0,data_preprocessed.csv,./data/data_preprocessed.csv,296
